In [1]:
import sys
import os
from pathlib import Path

# Detect environment and set working directory
try:
    import google.colab
    in_colab = True
    work_dir = Path("/content/auto-graph-rag/nations_demo_workspace")
    sys.path.append("/content/auto-graph-rag/src")
    print("🔍 Detected Google Colab environment")
except ImportError:
    in_colab = False
    work_dir = Path("./nations_demo_workspace")
    local_src = Path("./src")
    if local_src.exists():
        sys.path.append(str(local_src))
    print("🔍 Detected local environment")

# Create working directory
work_dir.mkdir(exist_ok=True)
print(f"📁 Working directory: {work_dir.absolute()}")

🔍 Detected local environment
📁 Working directory: /Users/benjaminfriedman/repos/auto-graph-rag/nations_demo_workspace


In [ ]:
# Set your API keys here
os.environ["OPENAI_API_KEY"] = ""  # Optional but recommended
os.environ["HF_TOKEN"] = ""      # Required

# Check environment
has_openai = bool(os.getenv("OPENAI_API_KEY")) and os.getenv("OPENAI_API_KEY") != "your-openai-api-key-here"
has_hf = bool(os.getenv("HF_TOKEN")) and os.getenv("HF_TOKEN") != "your-huggingface-token-here"

print(f"🔑 OpenAI API Key: {'✅ Available' if has_openai else '❌ Missing - Required for this demo'}")
print(f"🤗 HuggingFace Token: {'✅ Available' if has_hf else '⚠️ Missing - Recommended for model downloads'}")

if not has_openai:
    print("\n❌ Please set your OpenAI API key above to continue with the demo.")
    print("   You can get one at: https://platform.openai.com/api-keys")

🔑 OpenAI API Key: ✅ Available
🤗 HuggingFace Token: ✅ Available


In [3]:
import pykeen.datasets
import networkx as nx
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt

# Import Auto-Graph-RAG modules
from auto_graph_rag.modules import (
    GraphBuilder,
    GraphExplorer,
    DataGenerator,
    ModelTrainer,
    QueryExecutor
)

print("✅ All imports successful!")
print("Available modules:")
for module in [GraphBuilder, GraphExplorer, DataGenerator, ModelTrainer, QueryExecutor]:
    print(f"  - {module.__name__}")

HUGGINGFACE AVAILABLE
✅ All imports successful!
Available modules:
  - GraphBuilder
  - GraphExplorer
  - DataGenerator
  - ModelTrainer
  - QueryExecutor


In [4]:
def load_nations_dataset():
    """Load the Nations dataset from PyKEEN and convert to NetworkX."""
    print("📦 Loading Nations dataset from PyKEEN...")
    
    # Load dataset
    dataset = pykeen.datasets.Nations()
    training = dataset.training
    
    print(f"📊 Dataset Statistics:")
    print(f"  - Countries: {training.num_entities}")
    print(f"  - Relationship types: {training.num_relations}")
    print(f"  - Total relationships: {training.num_triples}")
    
    # Get mappings
    entity_to_id = training.entity_to_id
    relation_to_id = training.relation_to_id
    id_to_entity = {v: k for k, v in entity_to_id.items()}
    id_to_relation = {v: k for k, v in relation_to_id.items()}
    
    print(f"\n🌍 Countries in dataset:")
    countries = sorted(entity_to_id.keys())
    for i, country in enumerate(countries):
        if i % 5 == 0 and i > 0:
            print()  # New line every 5 countries
        print(f"{country:12}", end=" ")
    print()
    
    print(f"\n🔗 Top 10 relationship types:")
    triples = training.mapped_triples
    relation_counts = Counter()
    for i in range(len(triples)):
        h, r, t = triples[i]
        relation = id_to_relation[r.item()]
        relation_counts[relation] += 1
    
    for relation, count in relation_counts.most_common(10):
        print(f"  - {relation:20}: {count:3d} instances")
    
    # Create NetworkX graph
    print(f"\n🔄 Converting to NetworkX graph...")
    G = nx.DiGraph()
    
    # Add nodes (countries) with metadata
    for country in entity_to_id.keys():
        G.add_node(country, type="Country", name=country, region="Unknown")
    
    # Add edges (relationships)
    for i in range(len(triples)):
        h, r, t = triples[i]
        head_country = id_to_entity[h.item()]
        relation = id_to_relation[r.item()]
        tail_country = id_to_entity[t.item()]
        
        # Add edge with relation type
        G.add_edge(head_country, tail_country, type=relation, relation=relation)
    
    print(f"✅ NetworkX graph created:")
    print(f"  - Nodes: {G.number_of_nodes()}")
    print(f"  - Edges: {G.number_of_edges()}")
    print(f"  - Average degree: {sum(dict(G.degree()).values()) / G.number_of_nodes():.1f}")
    
    return G, relation_counts

# Load the dataset
nations_graph, relation_counts = load_nations_dataset()

📦 Loading Nations dataset from PyKEEN...
📊 Dataset Statistics:
  - Countries: 14
  - Relationship types: 55
  - Total relationships: 1592

🌍 Countries in dataset:
brazil       burma        china        cuba         egypt        
india        indonesia    israel       jordan       netherlands  
poland       uk           usa          ussr         

🔗 Top 10 relationship types:
  - embassy             : 100 instances
  - commonbloc1         :  80 instances
  - relngo              :  78 instances
  - relintergovorgs     :  74 instances
  - intergovorgs3       :  72 instances
  - timesinceally       :  70 instances
  - intergovorgs        :  67 instances
  - ngoorgs3            :  67 instances
  - reldiplomacy        :  66 instances
  - independence        :  64 instances

🔄 Converting to NetworkX graph...
✅ NetworkX graph created:
  - Nodes: 14
  - Edges: 182
  - Average degree: 26.0


In [5]:
# Initialize GraphBuilder
builder = GraphBuilder()

print("🔧 GraphBuilder Info:")
info = builder.get_info()
for key, value in info.items():
    print(f"  {key}: {value}")

print("\n📦 Building Nations graph database...")

# Extract node and edge labels
node_labels = {node: data["type"] for node, data in nations_graph.nodes(data=True)}
edge_labels = {(u, v): data["type"] for u, v, data in nations_graph.edges(data=True)}

print(f"🏷️ Creating labels:")
print(f"  - Node types: {set(node_labels.values())}")
print(f"  - Edge types: {len(set(edge_labels.values()))} unique relationship types")

# Build the database
db_path = work_dir / "nations_db"
stats = builder.build_from_networkx(
    graph=nations_graph,
    db_path=db_path,
    graph_name="nations",
    node_labels=node_labels,
    edge_labels=edge_labels
)

print("\n✅ Nations graph database created!")
print(f"📊 Database Statistics:")
for key, value in stats.items():
    if isinstance(value, list) and len(value) > 5:
        print(f"  {key}: {len(value)} items (showing first 5: {value[:5]}...)")
    else:
        print(f"  {key}: {value}")

🔧 GraphBuilder Info:
  name: GraphBuilder
  version: 1.0.0
  description: Build Kuzu graphs from NetworkX or raw data
  inputs: {'networkx': ['graph', 'db_path', 'graph_name', 'node_labels', 'edge_labels'], 'raw_data': ['nodes', 'edges', 'db_path', 'graph_name']}
  outputs: ['statistics', 'db_path']

📦 Building Nations graph database...
🏷️ Creating labels:
  - Node types: {'Country'}
  - Edge types: 24 unique relationship types

✅ Nations graph database created!
📊 Database Statistics:
  node_tables: ['nations_Country']
  edge_tables: 24 items (showing first 5: ['nations_timesinceally_Country_to_Country', 'nations_weightedunvote_Country_to_Country', 'nations_treaties_Country_to_Country', 'nations_commonbloc1_Country_to_Country', 'nations_unweightedunvote_Country_to_Country']...)
  total_nodes: 14
  total_edges: 182
  db_path: nations_demo_workspace/nations_db
  graph_name: nations


In [21]:
import openai
from auto_graph_rag.ingestion import KuzuAdapter

class OpenAIQueryExecutor:
    def __init__(self):
        # Initialize OpenAI client
        self.llm = openai.OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
        self.kuzu_graph_path = work_dir / "nations_db"
        
        # Use KuzuAdapter instead of direct Kuzu connection
        self.adapter = KuzuAdapter(self.kuzu_graph_path, create_new=False)  # Don't create new, connect to existing
        
        # Define a simple schema for the nations database
        # Since we know the structure, we can define it directly
        self.schema = """
Node Tables:
- nations_Country (properties: name, type, region)

Edge Tables (sample):
- nations_embassy_Country_to_Country
- nations_treaties_Country_to_Country  
- nations_commonbloc1_Country_to_Country
- nations_relngo_Country_to_Country
- nations_independence_Country_to_Country
- nations_violentactions_Country_to_Country
- nations_timesinceally_Country_to_Country
- nations_weightedunvote_Country_to_Country
... and more relationship types between countries
"""
        
        # Define the prompt template for GPT-4
        self.prompt_template = """You are a Cypher query expert for a Kuzu graph database.

The database contains the following schema:
{schema}

Given the user's question, generate a valid Cypher query that can be executed on the Kuzu graph to answer the question.

Rules:
1. Return ONLY the Cypher query without any explanation
2. Use the exact table names from the schema
3. Always include LIMIT 100 at the end of your query unless specifically counting
4. For node queries, the pattern is: MATCH (n:table_name) ...
5. For edge queries, the pattern is: MATCH (n:table1)-[r:edge_table]->(m:table2) ...

User Question: {question}

Cypher Query:"""
        
    def query_with_model(self, question):
        """Generate a Cypher query using GPT-4 and execute it on the Kuzu graph"""
        try:
            # Generate the prompt with schema and question
            prompt = self.prompt_template.format(
                schema=self.schema,
                question=question
            )
            
            # Call GPT-4 to generate the Cypher query
            response = self.llm.chat.completions.create(
                model="gpt-4",
                messages=[
                    {"role": "system", "content": "You are a Cypher query expert. Generate only valid Cypher queries."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,
                max_tokens=200
            )

            # Extract token usage
            usage = response.usage
        
        
            
            # Extract the generated query
            generated_query = response.choices[0].message.content.strip()
            
            # Clean up the query (remove markdown if present)
            if generated_query.startswith("```"):
                lines = generated_query.split("\n")
                generated_query = "\n".join(lines[1:-1])
            
            print(f"Generated Query: {generated_query}")
            
            # Execute the query using KuzuAdapter's connection
            result = self.adapter.conn.execute(generated_query)
            
            # Collect results
            results = []
            while result.has_next():
                row = result.get_next()
                results.append(row)
            
            return {
                "success": True,
                "query": generated_query,
                "results": results,
                "count": len(results),
                'tokens': {
                    'prompt_tokens': usage.prompt_tokens,
                    'completion_tokens': usage.completion_tokens,
                    'total_tokens': usage.total_tokens
                }
            }
            
        except Exception as e:
            return {
                "success": False,
                "error": str(e),
                "query": generated_query if 'generated_query' in locals() else None,
                'tokens': {
                    'prompt_tokens': 0,
                    'completion_tokens': 0,
                    'total_tokens': 0
                }
            }

In [39]:
# Initialize QueryExecutor
import timeit
import time
executor = QueryExecutor()
openai_executor = OpenAIQueryExecutor()


print("🔧 QueryExecutor Info:")
info = executor.get_info()
for key, value in info.items():
    print(f"  {key}: {value}")

# Use the trained model and database
model_path = work_dir / "nations_model"
print(f"\n🎯 Using Assets:")
print(f"  Model: {model_path} (Nations-specialized model)")
print(f"  Database: {db_path} (Nations knowledge graph)")

# International relations test questions
test_questions = [
    "What countries are in the dataset?",
    "Which countries have embassy relationships?",
    "What treaties exist between countries?",
    "What countries gained independance from china?",
    "Which country is in the most ngo relatinoships with other countries?",
    "Which countries does the USA have violent actions with?"
]

print(f"\n❓ Test Questions for International Relations:")
for i, question in enumerate(test_questions, 1):
    print(f"  {i}. {question}")

print(f"\n🤖 Query Execution with Nations-Specialized Model:")
print(f"Let's see how our model handles international relations queries!")
question_run_tracker = {}
question_run_tracker['openai'] = []
question_run_tracker['agr'] = []

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*60}")
    print(f"Query {i}: {question}")
    print(f"{'='*60}")

    start_agr = time.perf_counter()
    result = executor.query_with_model(
        question=question,
        model_path=model_path,
        db_path=db_path,
        return_cypher=True,
        format_results=True
    )
    end_agr = time.perf_counter()
    agr_time = end_agr - start_agr

    start_openai = time.perf_counter()
    openai_result = openai_executor.query_with_model(
        question=question
    )
    end_openai = time.perf_counter()

    openai_time = end_openai - start_openai    
        
    
    
    if result['success']:
        print(f"✅ Generated Cypher: {result['cypher']}")
        print(f"📊 Results: {result['count']} rows")
        
        if 'results' in result and result['results']:
            print(f"\n🔍 Sample Results:")
            # Show more results for international relations
            max_results = min(8, len(result['results']))
            for j, row in enumerate(result['results'][:max_results]):
                print(f"    {j+1:2d}. {row}")
            if len(result['results']) > max_results:
                print(f"    ... and {len(result['results']) - max_results} more")

        question_run_tracker['agr'].append({
            'question':question,
            'success':True,
            'time':agr_time,
            'prompt_tokens':0,
            'completion_tokens':0
        })
            
    else:
        print(f"❌ Error: {result['error']}")
        question_run_tracker['agr'].append({
                'question':question,
                'success':True,
                'time':agr_time,
                'prompt_tokens':0,
                'completion_tokens':0
            })


    if openai_result['success']:
        print(f"✅ OPENAI Generated Cypher: {result['cypher']}")
        # print(f"📊 Results: {result['count']} rows")
        
        if 'results' in result and openai_result['results']:
            print(f"\n🔍 Sample Results:")
            # Show more results for international relations
            max_results = min(8, len(result['results']))
            for j, row in enumerate(openai_result['results'][:max_results]):
                print(f"    {j+1:2d}. {row}")
            if len(openai_result['results']) > max_results:
                print(f"    ... and {len(openai_result['results']) - max_results} more")

        question_run_tracker['openai'].append({
            'question':question,
            'success':True,
            'time':openai_time,
            'prompt_tokens':openai_result['tokens']['prompt_tokens'],
            'completion_tokens':openai_result['tokens']['completion_tokens']
        })
        
    else:
        print(f"❌ OPENAI Error: {result['error']}")
        question_run_tracker['openai'].append({
                'question':question,
                'success':False,
                'time':openai_time,
                'prompt_tokens':0,
                'completion_tokens':0
            })
    
    
print(question_run_tracker)

Could not detect model name, using default


🔧 QueryExecutor Info:
  name: QueryExecutor
  version: 1.0.0
  description: Execute queries with fine-tuned models
  inputs: {'with_paths': ['question', 'model_path', 'db_path', 'return_cypher', 'format_results'], 'with_instances': ['question', 'model', 'kuzu_adapter', 'return_cypher', 'format_results']}
  outputs: ['success', 'cypher', 'results', 'count', 'error']

🎯 Using Assets:
  Model: nations_demo_workspace/nations_model (Nations-specialized model)
  Database: nations_demo_workspace/nations_db (Nations knowledge graph)

❓ Test Questions for International Relations:
  1. What countries are in the dataset?
  2. Which countries have embassy relationships?
  3. What treaties exist between countries?
  4. What countries gained independance from china?
  5. Which country is in the most ngo relatinoships with other countries?
  6. Which countries does the USA have violent actions with?

🤖 Query Execution with Nations-Specialized Model:
Let's see how our model handles international relat

What countries 
  are in the   
   dataset?    
┏━━━━━━━━━━━━━┓
┃ n.name      ┃
┡━━━━━━━━━━━━━┩
│ brazil      │
│ burma       │
│ china       │
│ cuba        │
│ egypt       │
│ india       │
│ indonesia   │
│ israel      │
│ jordan      │
│ netherlands │
│ poland      │
│ uk          │
│ usa         │
│ ussr        │
└─────────────┘

Could not detect model name, using default


Generated Query: MATCH (n:nations_Country) RETURN n.name LIMIT 100
✅ Generated Cypher: MATCH (n:nations_Country) RETURN n.name LIMIT 100
📊 Results: 14 rows

🔍 Sample Results:
     1. {'n.name': 'brazil'}
     2. {'n.name': 'burma'}
     3. {'n.name': 'china'}
     4. {'n.name': 'cuba'}
     5. {'n.name': 'egypt'}
     6. {'n.name': 'india'}
     7. {'n.name': 'indonesia'}
     8. {'n.name': 'israel'}
    ... and 6 more
✅ OPENAI Generated Cypher: MATCH (n:nations_Country) RETURN n.name LIMIT 100

🔍 Sample Results:
     1. ['brazil']
     2. ['burma']
     3. ['china']
     4. ['cuba']
     5. ['egypt']
     6. ['india']
     7. ['indonesia']
     8. ['israel']
    ... and 6 more

Query 2: Which countries have embassy relationships?


  Which   
countries 
   have   
 embassy  
relationsh
   ips?   
┏━━━━━━━━┓
┃ m.name ┃
┡━━━━━━━━┩
│ burma  │
│ cuba   │
│ jordan │
└────────┘

... (8 outputs left out)

Query failed (attempt 1): Binder exception: Table nations_nations_relngo_Country_to_Country does not exist.
Query failed (attempt 2): Binder exception: Table nations_nations_relngo_Country_to_Country does not exist.
Could not detect model name, using default


Generated Query: MATCH (n:nations_Country)-[r:nations_relngo_Country_to_Country]->(m:nations_Country)
RETURN n.name, COUNT(r) AS rel_count
ORDER BY rel_count DESC
LIMIT 1
❌ Error: Binder exception: Table nations_nations_relngo_Country_to_Country does not exist.
✅ OPENAI Generated Cypher: MATCH (n:nations_Country)-[r:nations_nations_relngo_Country_to_Country]->(m:nations_Country) RETURN m.name AS Country ORDER BY r.relation_count DESC LIMIT 1

Query 6: Which countries does the USA have violent actions with?


  Which   
countries 
 does the 
 USA have 
 violent  
 actions  
  with?   
┏━━━━━━━━┓
┃ o.name ┃
┡━━━━━━━━┩
│ china  │
└────────┘

Generated Query: MATCH (usa:nations_Country {name: 'USA'})-[r:nations_violentactions_Country_to_Country]->(otherCountry:nations_Country)
RETURN otherCountry.name
LIMIT 100
✅ Generated Cypher: MATCH (c:nations_Country)-[r:nations_violentactions_Country_to_Country]->(o:nations_Country) WHERE c.name = 'usa' RETURN o.name;
📊 Results: 1 rows

🔍 Sample Results:
     1. {'o.name': 'china'}
✅ OPENAI Generated Cypher: MATCH (c:nations_Country)-[r:nations_violentactions_Country_to_Country]->(o:nations_Country) WHERE c.name = 'usa' RETURN o.name;
{'openai': [{'question': 'What countries are in the dataset?', 'success': True, 'time': 1.363916333008092, 'prompt_tokens': 294, 'completion_tokens': 14}, {'question': 'Which countries have embassy relationships?', 'success': True, 'time': 1.898152250010753, 'prompt_tokens': 293, 'completion_tokens': 36}, {'question': 'What treaties exist between countries?', 'success': True, 'time': 2.40999866698985, 'prompt_tokens': 293, 'completion_tokens': 39}, {'que

In [40]:
import numpy as np
open_ai_time_mean = np.mean([r['time'] for r in question_run_tracker['openai']])
agr_time_mean = np.mean([r['time'] for r in question_run_tracker['agr']])

In [43]:
print(f"Mean OpenAI Inferance Time: {open_ai_time_mean} - Mean Auto Graph Rag Time: {agr_time_mean}")

Mean OpenAI Inferance Time: 2.159737423668654 - Mean Auto Graph Rag Time: 6.466415333163847
